In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy import signal, fft, stats
from scipy.ndimage import gaussian_filter
from scipy.stats import gaussian_kde
import os
import warnings
import cfad
from bg_atlasapi import BrainGlobeAtlas
import json
from matplotlib.colors import LogNorm
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from joblib import Parallel, delayed
from analysisUtils.preprocessing.processor import NeuropixelsProcessor
from analysisUtils.storage.funcs import save_scaled_lfp_to_dat, load_scaled_lfp_from_dat
from analysisUtils.helper import *
from analysisUtils.processing.funcs import *
from analysisUtils.processing.scRelated import *
from analysisUtils.visualisation.funcs import *
from analysisUtils.pipeline import *


# --- 1. GLOBAL CONFIGURATION ---
PARAMS = {
    # Hardware Specs
    'fs_raw': 30000,          # Acquisition Rate (Hz)
    'fs_lfp': 1000,           # Downsampled Rate (Hz)
    'voltage_scaling': 0.195, # uV per bit
    'probe_tip_offset': 720,  # Microns from tip with no channels

    # Analysis Windows
    'window_lfp': (-0.1, 0.5),    # Time window around trigger (s)
    'window_visual': (0.075, 0.150),# Window to find sSC sink (75 - 150ms post-stim)
    'bin_size_um': 15,            # Spatial bin for virtual probe
    
    # Bad Channel Thresholds
    'bad_ch_thresh': {
        'rms_low': 1.0,        # Dead (<1uV)
        'rms_high_factor': 15, # Noisy (>15x median)
        'corr_min': 0.6        # Artifact (<0.6 correlation to brain)
    },
    
    # Spike Statistics
    'spike_win_pre': (-0.2, 0.0), # Baseline window (s)
    'spike_win_post': (0, 0.2)# Visual response window (s)
}
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 14,
    'axes.labelsize': 16,
    'axes.titlesize': 18,
    'axes.linewidth': 1.5,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'pdf.fonttype': 42,  # Ensures fonts are editable in Adobe Illustrator
    'ps.fonttype': 42,
    'axes.spines.top': False,   # "Despining" for a cleaner look
    'axes.spines.right': False
})
Group = [None, "Vehicle", "Vehicle", "Psilocybin", "Vehicle", "Psilocybin", "Vehicle", "Psilocybin", "Psilocybin", "Vehicle", "Psilocybin", "Vehicle"]


In [3]:

if __name__ == "__main__":
    Mouse = []
    best_channels = []
    sc_depths = []
    latencies = []
    nVEPS = []
    mice = [2]
    for i in mice:
        print("Analysing Mouse " + str(i))
        FILE_PATH = cfad.data['Neural']['Renewal']['Concatenated Data']['Mouse ' + str(i)] + r"/mouse" + str(i) + r"_renewal_concatenated_neural_data.dat"
        KS_DIR = cfad.data['Neural']['Renewal']['Concatenated Data']['kilosort4']['Mouse ' + str(i)]
        checker_triggers = np.array(scipy.io.loadmat(cfad.data['Neural']['Renewal']['Triggers']['Checkerboard']['Mouse ' + str(i)])["evt"]).flatten()
        OUTPUT_FOLDER = r'Z:\Saij\Data\Checkerboard\ScMouse' + str(i) 
        TRIGGERS = checker_triggers[1:]
        probe_path = r"Z:\Saij\Herbs probes\Mouse " + str(i) + r"\probe_ccf_vox.npy" 
        probe = np.load(probe_path)
        probe *= 25
        
        results = run_sc_pipeline(FILE_PATH, KS_DIR, TRIGGERS, i, PARAMS, save_dir=OUTPUT_FOLDER, force_compute = False)
        depths = get_depth_map_full(KS_DIR)
        scID, scDepth, latency, volt = find_sc_channel(results["lfp"], PARAMS, results["anatomy"], results["depths"])
        Mouse.append(i)
        best_channels.append(scID)
        sc_depths.append(scDepth)
        latencies.append(latency)
        nVEPS.append(volt)

        fig, ax = plot_channel_sem(scID, results['lfp'], results['SEM'])
        save_pub_fig(fig, "10_Best_Channel", i, OUTPUT_FOLDER)
        df, info = map_probe_sc_centered(depths, scID, probe[0], probe[1])
    
        final_output_dir = r"Z:\Saij\ccf_channels"
        df.to_csv(final_output_dir + r"\Mouse " + str(i) + r".csv", index = False)
    Group = [None, "Vehicle", "Vehicle", "Psilocybin", "Vehicle", "Psilocybin", "Psilocybin", "Psilocybin", "Vehicle",]

"""
    vep_df = pd.DataFrame({
        "Mouse": Mouse,
        "Sc Channel": best_channels,
        "Depth (µm)": sc_depths,
        "Latency (s)": latencies,
        "nVEP (µV)": nVEPS,
        "Treatment Group": Group[2:]
    })

    vep_df.to_csv(r"Z:\Saij\Data\Checkerboard\nVEP.csv", index = False)
"""


Analysing Mouse 2
--- PROCESSING: mouse2_renewal_concatenated_neural_data.dat ---
Step 1: Loading Probe Geometry...
Step 2: Calibrating (Detecting Bad Channels)...
  > Flagged 5 Bad Channels
Step 3: Extracting Diagnostic Sample...
Step 4: Averaging 150 Trials...
Saved LFP binary to Z:\Saij\Data\Checkerboard\ScMouse2\data\meanLFP\meanLFP.dat
Saved LFP metadata to Z:\Saij\Data\Checkerboard\ScMouse2\data\meanLFP\meanLFP.json
Step 5: Creating Virtual Probe (Average)...
Saved LFP binary to Z:\Saij\Data\Checkerboard\ScMouse2\data\mean_virtualLFP\mean_virtualLFP.dat
Saved LFP metadata to Z:\Saij\Data\Checkerboard\ScMouse2\data\mean_virtualLFP\mean_virtualLFP.json
Step 6: Calculating CSD...
  > sSC Center: 3060.0 um
Step 7: Integrating Spikes...
Generating Standalone Plots...
Saved: Mouse2_1_Raw_Continuous
Saved: Mouse2_2_Virtual_Continuous
Saved: Mouse2_3_Avg_LFP
Saved: Mouse2_4_CSD
Saved: Mouse2_5_KDE_spikes
Calculating depth-resolved Spike Density MUA...
MUA Spiking map generated.
Saved: Mo

'\n    vep_df = pd.DataFrame({\n        "Mouse": Mouse,\n        "Sc Channel": best_channels,\n        "Depth (µm)": sc_depths,\n        "Latency (s)": latencies,\n        "nVEP (µV)": nVEPS,\n        "Treatment Group": Group[2:]\n    })\n\n    vep_df.to_csv(r"Z:\\Saij\\Data\\Checkerboard\nVEP.csv", index = False)\n'